In [4]:
# ============================================================
# Cell 1 : Configuration  ← only change NPZ_PATH
# ============================================================
from pathlib import Path

# Point this at wherever your .npz was saved by the training notebook
NPZ_PATH    = Path(r"M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_word_sequences_v2.npz")
LABELS_FILE = Path(r"M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\KARSL-502_Labels.txt")
CLASSES_CSV = Path(r"M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_v2_classes.csv")

# Expected values — the check will warn if these don't match
EXPECTED_CLASSES  = 502
EXPECTED_SEQ_LEN  = 48
EXPECTED_FEATURES = 258

print(f'NPZ     : {NPZ_PATH}')
print(f'Labels  : {LABELS_FILE}')
print(f'Classes : {CLASSES_CSV}')
print()
for name, p in [('NPZ', NPZ_PATH), ('Labels', LABELS_FILE), ('Classes CSV', CLASSES_CSV)]:
    exists = p.exists()
    size   = f'{p.stat().st_size / 1e6:.1f} MB' if exists else 'n/a'
    print(f'  [{"OK" if exists else "MISSING"}] {name}: {size}')

if not NPZ_PATH.exists():
    raise FileNotFoundError(
        f'NPZ not found:\n  {NPZ_PATH}\n\n'
        'Run Cells 0-7 of ArSL_Word_Training_v2.ipynb first to extract keypoints.'
    )


NPZ     : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_word_sequences_v2.npz
Labels  : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\KARSL-502_Labels.txt
Classes : M:\Term 10\Grad\SLR Main\Words\ArSL Word (Arabic)\arsl_v2_classes.csv

  [OK] NPZ: 135.6 MB
  [OK] Labels: 0.0 MB
  [OK] Classes CSV: 0.0 MB


In [5]:
# ============================================================
# Cell 2 : Load & Shape Check
# ============================================================
import numpy as np

print('Loading NPZ ...')
d = np.load(str(NPZ_PATH))
X = d['X']
y = d['y']

n_samples, seq_len, n_features = X.shape
n_classes = len(np.unique(y))

print()
print('=' * 50)
print('SHAPE SUMMARY')
print('=' * 50)
print(f'  Samples   : {n_samples:,}')
print(f'  Seq len   : {seq_len}')
print(f'  Features  : {n_features}')
print(f'  Classes   : {n_classes}')
print(f'  X dtype   : {X.dtype}')
print(f'  y dtype   : {y.dtype}')
print(f'  NPZ size  : {NPZ_PATH.stat().st_size / 1e6:.1f} MB')
print(f'  X memory  : {X.nbytes / 1e6:.0f} MB')
print()

# ── Validation ───────────────────────────────────────────────
ok = True
checks = [
    ('Classes',  n_classes,  EXPECTED_CLASSES,  '< 502 means extraction missed some classes'),
    ('Seq len',  seq_len,    EXPECTED_SEQ_LEN,  'mismatch with Kaggle notebook config'),
    ('Features', n_features, EXPECTED_FEATURES, 'mismatch — wrong feature extraction mode'),
]
print('VALIDATION')
print('-' * 50)
for name, got, expected, note in checks:
    status = 'OK' if got == expected else 'WARNING'
    print(f'  [{status}] {name}: {got}  (expected {expected})')
    if got != expected:
        print(f'         ^ {note}')
        ok = False

print()
if ok:
    print('All checks passed. NPZ looks correct for Kaggle.')
else:
    print('Fix the warnings above before uploading to Kaggle.')


Loading NPZ ...

SHAPE SUMMARY
  Samples   : 4,023
  Seq len   : 48
  Features  : 258
  Classes   : 502
  X dtype   : float32
  y dtype   : int32
  NPZ size  : 135.6 MB
  X memory  : 199 MB

VALIDATION
--------------------------------------------------
  [OK] Classes: 502  (expected 502)
  [OK] Seq len: 48  (expected 48)
  [OK] Features: 258  (expected 258)

All checks passed. NPZ looks correct for Kaggle.


In [6]:
# ============================================================
# Cell 3 : Class Distribution
# ============================================================
import pandas as pd

unique_ids, counts = np.unique(y, return_counts=True)

# Load English labels if available
id_to_english = {}
if LABELS_FILE.exists():
    with open(str(LABELS_FILE), 'r', encoding='utf-8', errors='replace') as fh:
        for line in fh:
            line = line.strip()
            if not line or line.lower().startswith('signid'):
                continue
            parts = line.split('\t')
            if len(parts) >= 3:
                try:
                    sid = int(parts[0])
                    en  = parts[2].strip()
                    id_to_english[sid + 1] = en if en not in ('', '?', '??') else str(sid + 1)
                except Exception:
                    pass

# Summary table (top 20 and bottom 20)
class_df = pd.DataFrame({
    'class_id': unique_ids,
    'english':  [id_to_english.get(int(uid), str(uid)) for uid in unique_ids],
    'count':    counts,
}).sort_values('count', ascending=False)

print('=' * 50)
print('CLASS DISTRIBUTION')
print('=' * 50)
print(f'  Total classes  : {len(unique_ids)}')
print(f'  Total samples  : {counts.sum():,}')
print(f'  Min / class    : {counts.min()}  ({class_df.iloc[-1]["english"]})')
print(f'  Max / class    : {counts.max()}  ({class_df.iloc[0]["english"]})')
print(f'  Mean / class   : {counts.mean():.1f}')
print(f'  Median / class : {np.median(counts):.0f}')
print(f'  Std dev        : {counts.std():.1f}')

# Classes with too few samples (risk of train split having 0 for that class)
thin = class_df[class_df['count'] < 5]
if not thin.empty:
    print(f'\n  WARNING: {len(thin)} classes have fewer than 5 samples:')
    print(thin.to_string(index=False))
else:
    print('\n  All classes have >= 5 samples (safe for stratified split).')

print('\nTop 10 most common classes:')
print(class_df.head(10).to_string(index=False))
print('\nBottom 10 least common classes:')
print(class_df.tail(10).to_string(index=False))


CLASS DISTRIBUTION
  Total classes  : 502
  Total samples  : 4,023
  Min / class    : 1  (4)
  Max / class    : 16  (700)
  Mean / class   : 8.0
  Median / class : 8
  Std dev        : 0.6

 class_id english  count
        6       4      1

Top 10 most common classes:
 class_id       english  count
       27           700     16
        5             3     15
      340  video camera      8
      334    chandelier      8
      335      cassette      8
      336 cassette tape      8
      337    television      8
      338     satellite      8
      339    video tape      8
        1             1      8

Bottom 10 least common classes:
 class_id     english  count
      168        rise      8
      167      inhale      8
      166     silence      8
      165        hear      8
      164     wake up      8
      163       sleep      8
      162       drink      8
      502   traveling      8
      105 thermometer      7
        6           4      1


In [7]:
# ============================================================
# Cell 4 : Feature & Sequence Quality Check
# ============================================================

print('=' * 50)
print('FEATURE STATISTICS')
print('=' * 50)

# Overall value range
print(f'  Global min     : {X.min():.4f}')
print(f'  Global max     : {X.max():.4f}')
print(f'  Global mean    : {X.mean():.4f}')
print(f'  Global std     : {X.std():.4f}')

# All-zero frames (frames where MediaPipe found nothing)
zero_frames   = np.sum(np.all(X == 0, axis=2))          # frames with all zero features
total_frames  = n_samples * seq_len
zero_pct      = zero_frames / total_frames * 100
print(f'\n  Zero frames    : {zero_frames:,} / {total_frames:,}  ({zero_pct:.1f}%)')
if zero_pct > 30:
    print('  WARNING: >30% zero frames — MediaPipe may not have detected hands/pose in many frames.')
elif zero_pct > 10:
    print('  NOTE: ~10-30% zero frames is normal for some poses or occlusions.')
else:
    print('  Zero frame rate looks healthy.')

# NaN / Inf check
n_nan = int(np.isnan(X).sum())
n_inf = int(np.isinf(X).sum())
print(f'\n  NaN values     : {n_nan}')
print(f'  Inf values     : {n_inf}')
if n_nan > 0 or n_inf > 0:
    print('  WARNING: NaN/Inf values will crash training. Re-run extraction.')
else:
    print('  No NaN or Inf values found.')

# Per-feature variance (very low variance = dead feature)
feat_var = X.reshape(-1, n_features).var(axis=0)
dead_feats = int((feat_var < 1e-6).sum())
print(f'\n  Dead features  : {dead_feats} / {n_features}  (variance < 1e-6)')
if dead_feats > 20:
    print('  WARNING: Many constant features detected. Check extraction pipeline.')


FEATURE STATISTICS
  Global min     : -1.4689
  Global max     : 1.7197
  Global mean    : 0.3344
  Global std     : 0.4220

  Zero frames    : 41,237 / 193,104  (21.4%)
  NOTE: ~10-30% zero frames is normal for some poses or occlusions.

  NaN values     : 0
  Inf values     : 0
  No NaN or Inf values found.

  Dead features  : 2 / 258  (variance < 1e-6)


In [8]:
# ============================================================
# Cell 5 : Visual Plots
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

sort_idx = np.argsort(counts)[::-1]
sorted_names  = [id_to_english.get(int(unique_ids[i]), str(unique_ids[i])) for i in sort_idx]
sorted_counts = counts[sort_idx]

fig, axes = plt.subplots(1, 3, figsize=(26, 5))

# ── Class distribution bar ─────────────────────────────────
axes[0].bar(range(len(sorted_names)), sorted_counts,
            color='#1565C0', edgecolor='none', alpha=0.85)
axes[0].axhline(counts.mean(),   color='red',    linestyle='--', alpha=0.8,
                label=f'Mean {counts.mean():.1f}')
axes[0].axhline(np.median(counts), color='orange', linestyle=':',  alpha=0.8,
                label=f'Median {np.median(counts):.0f}')
axes[0].set_title(f'Samples per Class  ({n_classes} classes, {n_samples:,} total)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Class (sorted by count)')
axes[0].set_ylabel('Samples')
axes[0].legend()

# ── Histogram ───────────────────────────────────────────────
axes[1].hist(sorted_counts, bins=min(30, n_classes),
             color='#1565C0', edgecolor='black', alpha=0.85)
axes[1].axvline(counts.mean(),     color='red',    linestyle='--', label=f'Mean {counts.mean():.1f}')
axes[1].axvline(np.median(counts), color='orange', linestyle=':',  label=f'Median {np.median(counts):.0f}')
axes[1].set_title('Distribution of Samples per Class', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Samples per class')
axes[1].set_ylabel('Number of classes')
axes[1].legend()

# ── Sample sequence visualisation ───────────────────────────
sample_idx = np.random.randint(0, n_samples)
sample_seq = X[sample_idx]   # (seq_len, n_features)
im = axes[2].imshow(sample_seq.T, aspect='auto', cmap='viridis',
                    interpolation='nearest')
plt.colorbar(im, ax=axes[2])
axes[2].set_title(f'Sample #{sample_idx}  —  class {y[sample_idx]} '
                  f'("{id_to_english.get(int(y[sample_idx]), "?")}")\n'
                  f'shape: ({seq_len}, {n_features})',
                  fontsize=11, fontweight='bold')
axes[2].set_xlabel('Frame')
axes[2].set_ylabel('Feature index')

plt.tight_layout()
plt.savefig('npz_check_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved: npz_check_plots.png')


Plot saved: npz_check_plots.png


C:\Users\adelg\AppData\Local\Temp\ipykernel_26048\905857448.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ============================================================
# Cell 6 : Kaggle Upload Readiness Verdict
# ============================================================

issues = []

if n_classes < EXPECTED_CLASSES:
    issues.append(f'Only {n_classes} classes found — expected {EXPECTED_CLASSES}. '
                  'Re-run extraction with full KARSL_ROOT path.')

if n_samples < 15000:
    issues.append(f'Only {n_samples:,} samples — expected ~25,111. '
                  'Extraction likely stopped early or used wrong path.')

if seq_len != EXPECTED_SEQ_LEN:
    issues.append(f'Sequence length {seq_len} != {EXPECTED_SEQ_LEN}. '
                  'Kaggle notebook Cell 2 SEQUENCE_LENGTH must match.')

if n_features != EXPECTED_FEATURES:
    issues.append(f'Feature count {n_features} != {EXPECTED_FEATURES}. '
                  'Kaggle notebook Cell 2 NUM_FEATURES must match.')

if n_nan > 0 or n_inf > 0:
    issues.append(f'{n_nan} NaN / {n_inf} Inf values found — re-run extraction.')

if not LABELS_FILE.exists():
    issues.append('KARSL-502_Labels.txt missing — also upload this to Kaggle.')

if not CLASSES_CSV.exists():
    issues.append('arsl_v2_classes.csv missing — also upload this to Kaggle.')

print('=' * 55)
print('KAGGLE UPLOAD READINESS')
print('=' * 55)

if not issues:
    print('READY TO UPLOAD')
    print()
    print('Upload these 3 files as a Kaggle dataset:')
    print(f'  1. {NPZ_PATH.name}    ({NPZ_PATH.stat().st_size / 1e6:.0f} MB)')
    if LABELS_FILE.exists():
        print(f'  2. {LABELS_FILE.name}  ({LABELS_FILE.stat().st_size / 1024:.0f} KB)')
    if CLASSES_CSV.exists():
        print(f'  3. {CLASSES_CSV.name}      ({CLASSES_CSV.stat().st_size / 1024:.0f} KB)')
    print()
    print(f'Then set DATASET_SLUG in Kaggle Cell 2 and run all cells.')
else:
    print('NOT READY — fix these issues first:')
    for i, issue in enumerate(issues, 1):
        print(f'  {i}. {issue}')


KAGGLE UPLOAD READINESS
NOT READY — fix these issues first:
  1. Only 4,023 samples — expected ~25,111. Extraction likely stopped early or used wrong path.
